In [1]:
import pandas as pd
import numpy as np
import re
import difflib

# 1. Carga de archivos
customers_raw = pd.read_csv('pedido_standard_customers.csv')
orders_raw = pd.read_csv('pedido_standard_orders.csv')
products_raw = pd.read_csv('pedido_standard_products.csv')

# 2. Limpieza de Clientes
customers = customers_raw.copy()

def clean_name(val):
    if pd.isna(val): return np.nan
    s = re.sub(r'\s+', ' ', str(val).strip())
    return s.title() if s else np.nan

def clean_email(val):
    if pd.isna(val): return np.nan
    s = re.sub(r'\s+', '', str(val).strip().lower())
    return s if s else np.nan

def clean_city(val):
    if pd.isna(val): return np.nan
    s = re.sub(r'\s+', ' ', str(val).strip().lower())
    s_no_acc = s.replace('ó', 'o').replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ú', 'u')
    if s_no_acc in ['buenos aires', 'bs. as.', 'bs as', 'caba']:
        return 'Buenos Aires'
    elif s_no_acc == 'cordoba':
        return 'Córdoba'
    elif s_no_acc == 'rosario':
        return 'Rosario'
    elif s_no_acc == 'mendoza':
        return 'Mendoza'
    return np.nan

customers['name'] = customers['name'].apply(clean_name)
customers['email'] = customers['email'].apply(clean_email)
customers['city'] = customers['city'].apply(clean_city)

# Fusión de duplicados por email conservando el primer valor no nulo
customers_clean = customers.groupby('email', as_index=False, sort=False).first()[['name', 'email', 'city']]

# 3. Limpieza y Deduplicación de Pedidos
orders = orders_raw.copy()
orders['order_id'] = orders['order_id'].astype(str).str.strip().str.upper().str.replace(r'\s+', '', regex=True)
orders['updated_at_dt'] = pd.to_datetime(orders['updated_at'], errors='coerce')

# Deduplicar quedándose con el registro de updated_at más reciente
orders_dedup = (
    orders.sort_values('updated_at_dt', ascending=True, kind='mergesort')
    .drop_duplicates(subset=['order_id'], keep='last')
    .sort_values('order_id')
    .reset_index(drop=True)
)

def clean_order_date(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', s):
        dt = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    elif re.match(r'^\d{1,2}[/-]\d{1,2}[/-]\d{4}$', s):
        dt = pd.to_datetime(s, dayfirst=True, errors='coerce')
    else:
        dt = pd.to_datetime(s, errors='coerce')
    return dt.strftime('%Y-%m-%d') if pd.notna(dt) else np.nan

orders_dedup['order_date'] = orders_dedup['order_date'].apply(clean_order_date)
orders_dedup['customer_email'] = orders_dedup['customer_email'].apply(clean_email)

# Normalización de productos contra el catálogo de 12 ítems
catalog = products_raw['product'].str.strip().tolist()
catalog_lower = [c.lower() for c in catalog]

def clean_product(val):
    if pd.isna(val): return np.nan
    s = re.sub(r'\s+', ' ', str(val).strip())
    if s.lower() in catalog_lower:
        return catalog[catalog_lower.index(s.lower())]
    matches = difflib.get_close_matches(s.lower(), catalog_lower, n=1, cutoff=0.6)
    return catalog[catalog_lower.index(matches[0])] if matches else s

orders_dedup['product'] = orders_dedup['product'].apply(clean_product)

# Normalización de cantidad y precio unitario
def clean_quantity(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()
    word_map = {'uno': 1, 'dos': 2, 'tres': 3}
    if s in word_map: return word_map[s]
    try:
        return int(float(s))
    except:
        return np.nan

def clean_price(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().replace('$', '').replace(' ', '')
    if not s or s.lower() in ['nan', 'none', 'null']: return np.nan
    if ',' in s:
        s = s.replace('.', '').replace(',', '.')
    try:
        return round(float(s), 2)
    except:
        return np.nan

def clean_status(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()
    status_map = {
        'entregado': 'Entregado', 'delivered': 'Entregado',
        'enviado': 'Enviado', 'shipped': 'Enviado',
        'pendiente': 'Pendiente', 'pending': 'Pendiente',
        'cancelado': 'Cancelado', 'canceled': 'Cancelado', 'cancelled': 'Cancelado'
    }
    return status_map.get(s, s.title())

orders_dedup['quantity'] = orders_dedup['quantity'].apply(clean_quantity).astype('Int64')
orders_dedup['unit_price'] = orders_dedup['unit_price'].apply(clean_price)
orders_dedup['status'] = orders_dedup['status'].apply(clean_status)

# 4. Aplicación de reglas de rechazo en orden estricto
def get_rejection_reason(row):
    if pd.isna(row['quantity']) or row['quantity'] <= 0 or row['quantity'] > 50:
        return 'invalid_quantity'
    if pd.notna(row['order_date']) and row['order_date'] > '2026-11-13':
        return 'future_date'
    if pd.isna(row['unit_price']):
        return 'missing_price'
    return np.nan

orders_dedup['reason'] = orders_dedup.apply(get_rejection_reason, axis=1)

orders_rejected = orders_dedup[orders_dedup['reason'].notna()][['order_id', 'reason']].copy().reset_index(drop=True)
orders_clean = orders_dedup[orders_dedup['reason'].isna()].copy().reset_index(drop=True)

# 5. Columnas calculadas y exportación
orders_clean['total'] = (orders_clean['quantity'] * orders_clean['unit_price']).round(2)
valid_emails = set(customers_clean['email'].dropna())
orders_clean['customer_found'] = orders_clean['customer_email'].isin(valid_emails)

orders_clean = orders_clean[['order_id', 'order_date', 'customer_email', 'product', 'quantity', 'unit_price', 'total', 'status', 'customer_found']]

customers_clean.to_csv('customers_clean.csv', index=False)
orders_clean.to_csv('orders_clean.csv', index=False, float_format='%.2f')
orders_rejected.to_csv('orders_rejected.csv', index=False)